# Nonlinear Grassmannian Analysis: MLP Predictor
### Companion to *The Geometry of the Virtue of Complexity in Asset Pricing: Nonlinear Predictive Models*
#### Deep, Lesniewski, Missaoui, Pakala (2026)

**Core object**: the stacked-gradient matrix $G_\\theta(Z_t) \\in \\mathrm{Mat}_{N,m_0}(\\mathbb{R})$, row $i$ = $\\nabla f_\\theta(Z_{i,t})$ — NOT the full Jacobian (which degenerates for pooled architectures, see paper Section 3).

**Complexity axis**: hidden width $K \\in \\{16, 32, 64, 128\\}$.

In [ ]:
# Run once if torch is not installed:
# !pip install torch --index-url https://download.pytorch.org/whl/cpu


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

ROOT_DIR = Path('..').resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.mlp import CrossSectionalMLP, train_mlp
from src.jacobian_grass import (
    stacked_gradient_matrix, grassmann_subspace, principal_angles,
    projection_distance, grassmann_velocity, anisotropy_ratio,
    spectral_gap, predictive_alignment, geometric_alpha, smoothed_subspace,
)

print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
print(f'ROOT_DIR: {ROOT_DIR}')


## 1. Data Loading
Same 50 stocks × 73 characteristics as the companion IPCA-RFF paper.

In [ ]:
DATA_DIR = ROOT_DIR / 'data'
df_char_filtered = pd.read_parquet(DATA_DIR / 'ipca_char_data_filtered.parquet')
print(f'Full panel: {df_char_filtered.shape}  |  permnos: {df_char_filtered.permno.nunique()}')


In [ ]:
# Load cached top-50 permnos; if missing, re-run WRDS query from experimentation_IPCA_rff.ipynb cells 21-24
TOP50_CACHE = DATA_DIR / 'top50_permnos.csv'

if TOP50_CACHE.exists():
    top50_permnos = pd.read_csv(TOP50_CACHE)['permno'].tolist()
    print(f'Loaded top-50 permnos from cache')
else:
    print('Cache missing — run WRDS query from experimentation_IPCA_rff.ipynb cells 21-24,\n'
          'then save: pd.DataFrame({"permno": top50_permnos}).to_csv(TOP50_CACHE, index=False)')
    raise FileNotFoundError(f'{TOP50_CACHE} not found. See comment above.')

df_top50 = df_char_filtered[df_char_filtered.permno.isin(top50_permnos)].copy()
print(f'df_top50: {df_top50.shape}  |  permnos: {df_top50.permno.nunique()}')


In [ ]:
df = df_top50.copy()
df['date'] = pd.to_datetime(df['yyyymm'])
df = df.drop(columns=['yyyymm']).set_index(['date', 'permno']).sort_index()

if 'y_ipca' not in df.columns:
    df['y_ipca'] = df.groupby(level='permno')['excess_ret'].shift(-1)

char_cols = [c for c in df.columns if c not in ('y_ipca', 'excess_ret', 'Price')]
print(f'char_cols ({len(char_cols)}): {char_cols[:6]} ...')
print(f'Panel: {df.shape}  |  dates: {df.index.get_level_values("date").nunique()}')


## 2. Cross-Sectional Tensors
Build $(Z_t, r_t)$ pairs with cross-sectional rank normalisation.

In [ ]:
def build_panel_tensors(df, char_cols, ret_col='y_ipca'):
    dates, Z_list, r_list = [], [], []
    for date, grp in df.groupby(level='date'):
        r    = grp[ret_col].values.astype(np.float32)
        Z_raw= grp[char_cols].values.astype(np.float32)
        mask = np.isfinite(r) & np.all(np.isfinite(Z_raw), axis=1)
        if mask.sum() < 5:
            continue
        r, Z_raw = r[mask], Z_raw[mask]
        N = Z_raw.shape[0]
        ranks = np.argsort(np.argsort(Z_raw, axis=0), axis=0).astype(np.float32)
        Z_norm = ranks / max(N - 1, 1) - 0.5   # rank-normalise to [-0.5, 0.5]
        dates.append(date)
        Z_list.append(torch.tensor(Z_norm))
        r_list.append(torch.tensor(r))
    return dates, Z_list, r_list

dates, Z_list, r_list = build_panel_tensors(df, char_cols)
N_ASSETS = int(np.median([z.shape[0] for z in Z_list]))
M0 = len(char_cols)
T  = len(dates)
print(f'T={T}  N~{N_ASSETS}  m0={M0}')
print(f'{dates[0].date()} -> {dates[-1].date()}')


## 3. Smoke Test: Stacked-Gradient Matrix $G_\\theta(Z_t)$

Verify shape, rank, and all diagnostics on a randomly initialised model.

In [ ]:
K_test = 32; k_test = 4; t_idx = T // 2
model_test = CrossSectionalMLP(m0=M0, K=K_test)
Z_t = Z_list[t_idx]; r_t = r_list[t_idx].numpy()

G_t = stacked_gradient_matrix(model_test, Z_t)
print(f'G shape : {G_t.shape}  (expected ({Z_t.shape[0]}, {M0}))')
print(f'G rank  : {np.linalg.matrix_rank(G_t)}')

U_k, sigma, erank = grassmann_subspace(G_t, k=k_test)
print(f'sigma[:6]: {sigma[:6].round(3)}')
print(f'erank    : {erank:.2f}  (1=concentrated, m0=diffuse)')

# Dummy previous subspace for motion diagnostics
G_prev = stacked_gradient_matrix(CrossSectionalMLP(m0=M0, K=K_test), Z_t)
U_prev, _, _ = grassmann_subspace(G_prev, k=k_test)
n = min(U_k.shape[0], U_prev.shape[0])
ang = principal_angles(U_k[:n], U_prev[:n])
print(f'angles (deg)  : {np.degrees(ang).round(1)}')
print(f'd_proj        : {projection_distance(U_k[:n], U_prev[:n]):.4f}')
print(f'velocity v_t  : {grassmann_velocity(U_k[:n], U_prev[:n]):.4f}')
print(f'rho_theta     : {anisotropy_ratio(ang):.4f}')
print(f'spectral gap  : {spectral_gap(sigma, k_test):.4f}')
print(f'eta (alignment): {predictive_alignment(U_k, r_t):.4f}')
alpha = geometric_alpha(U_k, r_t)
print(f'||alpha^geo||/||r||: {np.linalg.norm(alpha)/(np.linalg.norm(r_t)+1e-9):.4f}')


## 4. Save top-50 permno cache
Run this once after loading `df_top50` from `experimentation_IPCA_rff.ipynb`.

In [ ]:
# Run from experimentation_IPCA_rff.ipynb after cell 24 to save the cache:
# pd.DataFrame({'permno': top50_permnos}).to_csv(DATA_DIR / 'top50_permnos.csv', index=False)
# print('Saved top50_permnos.csv')


## 5. Rolling-Window Backtest

For each step $t$:
1. Train MLP on $[t-T_{\\mathrm{win}}, t)$
2. Predict $\\hat{r}_t$ → compute OOS $R^2$ and portfolio return
3. Extract $G_\\theta(Z_t)$, subspace $V_t^{(k)}$, all diagnostics
4. Optionally smooth via $M_t = \\frac{1}{w}\\sum G_s G_s^\\top$

In [ ]:
from dataclasses import dataclass, field

@dataclass
class MLPResult:
    K: int; z: float; k: int; window_len: int
    dates:        list = field(default_factory=list)
    r2_oos:       list = field(default_factory=list)
    sharpe_tick:  list = field(default_factory=list)
    d_proj:       list = field(default_factory=list)
    v_geo:        list = field(default_factory=list)
    kappa:        list = field(default_factory=list)
    theta_max:    list = field(default_factory=list)
    theta_bar:    list = field(default_factory=list)
    rho_theta:    list = field(default_factory=list)
    erank:        list = field(default_factory=list)
    sg:           list = field(default_factory=list)
    eta:          list = field(default_factory=list)   # predictive alignment
    alpha_norm:   list = field(default_factory=list)   # ||alpha^geo|| / ||r||


In [ ]:
def run_mlp_grass(dates, Z_list, r_list,
                  K=32, z=10.0, k=4, window_len=24, L=2,
                  lr=1e-3, max_epochs=200, smooth_w=3, verbose=True):
    res = MLPResult(K=K, z=z, k=k, window_len=window_len)
    m0 = Z_list[0].shape[1]
    U_prev, v_prev, G_buf = None, None, []

    for t in range(window_len, len(dates)):
        # training window
        Z_tr = torch.cat(Z_list[t - window_len:t], dim=0)
        r_tr = torch.cat(r_list[t - window_len:t], dim=0)
        mask_tr = torch.isfinite(r_tr) & torch.all(torch.isfinite(Z_tr), dim=1)
        if mask_tr.sum() < k + 2: continue
        Z_tr, r_tr = Z_tr[mask_tr], r_tr[mask_tr]

        model = CrossSectionalMLP(m0=m0, K=K, L=L)
        train_mlp(model, Z_tr, r_tr, z=z, lr=lr, max_epochs=max_epochs)

        # test step
        Z_t = Z_list[t]
        r_np = r_list[t].numpy()
        mask_t = np.isfinite(r_np) & np.all(np.isfinite(Z_t.numpy()), axis=1)
        if mask_t.sum() < k + 2: continue
        Z_tc, r_tc = Z_t[mask_t], r_np[mask_t]

        with torch.no_grad(): r_hat = model(Z_tc).numpy()
        ss_res = ((r_tc - r_hat)**2).sum()
        ss_tot = ((r_tc - r_tc.mean())**2).sum()
        r2 = float(1 - ss_res / (ss_tot + 1e-12))

        # long-short portfolio
        rank = np.argsort(r_hat)
        h = len(rank) // 2
        port_ret = r_tc[rank[h:]].mean() - r_tc[rank[:h]].mean()

        # G_theta, subspace
        G_t = stacked_gradient_matrix(model, Z_tc)
        U_k, sigma, erank_val = grassmann_subspace(G_t, k=k)

        # smoothed subspace
        G_buf.append(G_t)
        if len(G_buf) > smooth_w: G_buf.pop(0)

        sg_val   = spectral_gap(sigma, k)
        eta_val  = predictive_alignment(U_k, r_tc)
        a_norm   = float(np.linalg.norm(geometric_alpha(U_k, r_tc)) / (np.linalg.norm(r_tc) + 1e-12))

        if U_prev is not None:
            n = min(U_k.shape[0], U_prev.shape[0])
            ang    = principal_angles(U_k[:n], U_prev[:n])
            d_proj = projection_distance(U_k[:n], U_prev[:n])
            v_t    = grassmann_velocity(U_k[:n], U_prev[:n])
            kappa  = abs(v_t - v_prev) if v_prev is not None else 0.0
            res.dates.append(dates[t])
            res.r2_oos.append(r2); res.sharpe_tick.append(float(port_ret))
            res.d_proj.append(d_proj); res.v_geo.append(v_t); res.kappa.append(kappa)
            res.theta_max.append(float(ang[0])); res.theta_bar.append(float(np.mean(ang)))
            res.rho_theta.append(anisotropy_ratio(ang))
            res.erank.append(erank_val); res.sg.append(sg_val)
            res.eta.append(eta_val); res.alpha_norm.append(a_norm)
            v_prev = v_t
        else:
            v_prev = None
        U_prev = U_k

        if verbose and len(res.dates) % 12 == 0:
            print(f'  {dates[t].date()}  r2={r2:+.3f}  eta={eta_val:.3f}  rho={res.rho_theta[-1] if res.rho_theta else 0:.3f}')
    return res


## 6. Quick Single-Config Run

In [ ]:
print('Running: K=32, z=10, k=4, window=24')
res = run_mlp_grass(dates, Z_list, r_list,
                    K=32, z=10.0, k=4, window_len=24, verbose=True)
print(f'\nOOS steps      : {len(res.dates)}')
print(f'Mean OOS R2    : {np.mean(res.r2_oos):.4f}')
sharpe = np.mean(res.sharpe_tick) / (np.std(res.sharpe_tick) + 1e-9) * np.sqrt(12)
print(f'Sharpe (ann)   : {sharpe:.3f}')
print(f'Mean rho_theta : {np.mean(res.rho_theta):.4f}')
print(f'Mean eta       : {np.mean(res.eta):.4f}')
print(f'Mean erank     : {np.mean(res.erank):.2f}')


## 7. Diagnostic Time Series

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 9))
fig.suptitle(f'MLP diagnostics  K={res.K}  z={res.z}  k={res.k}  win={res.window_len}m', fontsize=11)
ts = pd.DatetimeIndex(res.dates)
panels = [
    (res.r2_oos,         'OOS R2',           'steelblue'),
    (res.sharpe_tick,    'Portfolio ret',     'green'),
    (res.d_proj,         'd_proj',            'darkorange'),
    (res.v_geo,          'Velocity v_t',      'purple'),
    (res.kappa,          'Curvature kappa_t', 'red'),
    (res.rho_theta,      'rho_theta',         'brown'),
    (res.erank,          'Effective rank',    'teal'),
    (res.eta,            'eta (alignment)',   'darkblue'),
    (res.alpha_norm,     '||alpha_geo||/||r||','crimson'),
]
for ax, (vals, title, col) in zip(axes.flat, panels):
    ax.plot(ts, vals, color=col, lw=0.8)
    ax.axhline(np.mean(vals), color='k', ls='--', lw=0.6, alpha=0.5)
    ax.set_title(title, fontsize=9); ax.tick_params(labelsize=7)
plt.tight_layout()
out = ROOT_DIR / 'plots' / 'nonlinear_MLP_diagnostics.pdf'
plt.savefig(out, bbox_inches='tight'); plt.show()
print(f'Saved {out}')


## 8. Width Sweep $K \\in \\{16, 32, 64, 128\\}$

Expected (virtue-of-complexity): as $K$ increases,
$\\eta_{t+1}\\uparrow$, $v_t\\uparrow$, $\\rho_\\theta\\downarrow$.

In [ ]:
# ~15-30 min on CPU  (uncomment to run)
# K_VALUES = [16, 32, 64, 128]
# sweep = {}
# for K in K_VALUES:
#     print(f'K={K}')
#     sweep[K] = run_mlp_grass(dates, Z_list, r_list, K=K, z=10.0, k=4,
#                              window_len=24, verbose=False)
#     r = sweep[K]
#     sh = np.mean(r.sharpe_tick) / (np.std(r.sharpe_tick)+1e-9) * np.sqrt(12)
#     print(f'  R2={np.mean(r.r2_oos):.4f}  Sharpe={sh:.3f}  eta={np.mean(r.eta):.4f}  rho={np.mean(r.rho_theta):.4f}')


In [ ]:
# Summary table + velocity-alignment scatter  (run after sweep above)
# rows = []
# for K, r in sweep.items():
#     sh = np.mean(r.sharpe_tick) / (np.std(r.sharpe_tick)+1e-9) * np.sqrt(12)
#     rows.append({'K':K, 'OOS R2':round(np.mean(r.r2_oos),4),
#                  'Sharpe':round(sh,3), 'v_geo':round(np.mean(r.v_geo),4),
#                  'rho_theta':round(np.mean(r.rho_theta),4),
#                  'erank':round(np.mean(r.erank),2), 'eta':round(np.mean(r.eta),4)})
# print(pd.DataFrame(rows).set_index('K').to_string())
